In [1]:
!pip install transformers datasets peft accelerate trl bitsandbytes


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

In [2]:
import torch
import json
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig # type: ignore
from huggingface_hub import login
import tqdm



/Users/philippplotnikov/.pyenv/versions/3.11.8/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 60
print(f"Используем устройство: {DEVICE}")

Используем устройство: cpu


In [ ]:
COLAB_HUGGING_FACE_ACCESS_TOKEN = "" # your token
login(token=COLAB_HUGGING_FACE_ACCESS_TOKEN)

In [4]:
model_path = "../models/qwen3-8b"

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map={"": "mps"},
    trust_remote_code=True
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if tokenizer.bos_token_id is None:
    tokenizer.bos_token = tokenizer.eos_token

# ЯВНО синхронизируем конфиги модели с токенизатором
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

base_model.config.eos_token_id = tokenizer.eos_token_id
base_model.generation_config.eos_token_id = tokenizer.eos_token_id

base_model.config.bos_token_id = tokenizer.bos_token_id
base_model.generation_config.bos_token_id = tokenizer.bos_token_id

Loading weights: 100%|██████████| 399/399 [00:16<00:00, 24.46it/s]


In [5]:
with open("../data/summarization_dataset.json", "r", encoding="utf-8") as f:
    raw_dataset = json.load(f)

temp_data = []
for item in raw_dataset:
    system_query = {
        "role": "system",
        "content": (
            "Обнови краткий контекст диалога.\n\n"
            "Правила:\n"
            "- Сохраняй только факты\n"
            "- Убирай дубли\n"
            "- Сжимай текст\n"
            "- Сохраняй ингредиенты, названия блюд и действия\n"
            "- Итог: краткий (до 7 предложений)\n\n"
            "Предыдущий диалог:\n"
            f"{item['previous_text']}\n\n"
            "Текущий диалог:\n"
            f"{item['current_text']}\n\n"
            "Напиши обновлённый краткий контекст."
        )
    }
    assistant_response = {
        "role": "assistant",
        "content": item['summary']
    }
    temp_data.append({'messages': [system_query, assistant_response]})
dataset = Dataset.from_list(temp_data)

In [6]:
dataset[0]

{'messages': [{'content': 'Обнови краткий контекст диалога.\n\nПравила:\n- Сохраняй только факты\n- Убирай дубли\n- Сжимай текст\n- Сохраняй ингредиенты, названия блюд и действия\n- Итог: краткий (до 7 предложений)\n\nПредыдущий диалог:\nПользователь обсуждает Куриный суп с лапшой. Используются ингредиенты: курица, лапша, морковь, лук. Пользователь обсуждает Ризотто с грибами. Используются ингредиенты: рис, грибы, бульон, сыр.\n\nТекущий диалог:\nЯ тут делаю Борщ с говядиной, что дальше делать?\n\nНапиши обновлённый краткий контекст.',
   'role': 'system'},
  {'content': 'Обсуждаются блюда: Куриный суп с лапшой: используются курица, лапша, морковь, лук; Ризотто с грибами: используются рис, грибы, бульон, сыр; Борщ с говядиной: используются говядина, свекла, капуста, картофель; пользователь уточняет приготовление.',
   'role': 'assistant'}]}

In [7]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
)



In [8]:
training_args = SFTConfig(
    output_dir="../models/qwen3-8b-summarization-sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=1,
    logging_steps=5,
    save_steps=50,
    bf16=True,
    report_to="none",
    assistant_only_loss=True,
    gradient_checkpointing=True,
    dataset_text_field="messages",
    shuffle_dataset=True,
    packing=True,
)


In [12]:
def formatting_func(example):
    return example


In [13]:
train_dataset = dataset.select(range(200))
eval_dataset = dataset.select(range(200, 300))

In [14]:
trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    processing_class=tokenizer,
    peft_config=lora_config,
    formatting_func=formatting_func
)

print("--- Проверяем один обработанный пример ---")

processed_example = trainer.train_dataset[1] # type: ignore
input_ids = processed_example['input_ids']

print(f"Количество токенов в input_ids: {len(input_ids)}")

print("\n[Полный вход (input_ids), декодированный]:")
print(tokenizer.decode(input_ids))



/Users/philippplotnikov/.pyenv/versions/3.11.8/lib/python3.11/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/philippplotnikov/.pyenv/versions/3.11.8/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementat

--- Проверяем один обработанный пример ---
Количество токенов в input_ids: 1023

[Полный вход (input_ids), декодированный]:
<|im_start|>system
Обнови краткий контекст диалога.

Правила:
- Сохраняй только факты
- Убирай дубли
- Сжимай текст
- Сохраняй ингредиенты, названия блюд и действия
- Итог: краткий (до 7 предложений)

Предыдущий диалог:
Пользователь обсуждает Блины с творогом. Используются ингредиенты: мука, яйца, молоко, творог. Пользователь обсуждает Блины с творогом. Используются ингредиенты: мука, яйца, молоко, творог. Пользователь обсуждает Омлет с сыром. Используются ингредиенты: яйца, сыр, молоко. Пользователь обсуждает Тушеная говядина с овощами. Используются ингредиенты: говядина, картофель, морковь, лук. Пользователь обсуждает Ризотто с грибами. Используются ингредиенты: рис, грибы, бульон, сыр. Пользователь обсуждает Плов с курицей. Используются ингредиенты: рис, курица, морковь, лук.

Текущий диалог:
Начал готовить Жареная картошка с грибами, что следующим шагом?

Напи

In [ ]:
trainer.train()
trainer.save_model("../models/qwen3-8b-summarization-sft/checkpoint-final")

/Users/philippplotnikov/.pyenv/versions/3.11.8/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,2.274447
10,1.215226
15,0.900803


TrainOutput(global_step=19, training_loss=1.3225042067076032, metrics={'train_runtime': 503.3369, 'train_samples_per_second': 0.298, 'train_steps_per_second': 0.038, 'total_flos': 1703884965040128.0, 'train_loss': 1.3225042067076032})

In [32]:
model = trainer.model
model = model.merge_and_unload() # type: ignore

In [ ]:
text = tokenizer.apply_chat_template(
    [
        {
            "role": "user",
            "content": (
                "Обнови краткий контекст диалога.\n\n"
                "Правила:\n"
                "- Сохраняй только факты\n"
                "- Убирай дубли\n"
                "- Сжимай текст\n"
                "- Сохраняй ингредиенты, названия блюд и действия\n"
                "- Итог: краткий (до 7 предложений)\n\n"
                "Предыдущий диалог:\n"
                "Пользователь уточнил, что готовый форша-мат по-фински из говядины и сельди хранится 2 дня. Второй пользователь спрашивает о части рыбы в рецепте.\n\n"
                "Текущий диалог:\n"
                "Я сейчас захотел тонкие блины на сыворотке с содой.  Подскажи, какая жидкость используется в рецепте для приготовления тонких блинов\n\n"
                "Напиши обновлённый краткий контекст."
            )
        },
    ],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
text

'<|im_start|>user\nОбнови краткий контекст диалога.\n\nПравила:\n- Сохраняй только факты\n- Убирай дубли\n- Сжимай текст\n- Сохраняй ингредиенты, названия блюд и действия\n- Итог: краткий (до 7 предложений)\n\nПредыдущий диалог:\nПользователь уточнил, что готовый форша-мат по-фински из говядины и сельди хранится 2 дня. Второй пользователь спрашивает о части рыбы в рецепте.\n\nТекущий диалог:\nЯ сейчас захотел тонкие блины на сыворотке с содой.  Подскажи, какая жидкость используется в рецепте для приготовления тонких блинов\n\nНапиши обновлённый краткий контекст.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [ ]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)


model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True
with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=False,  # для диагностики лучше так
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )

output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")

print(content)

content: Пользователь уточнил, что готовый форша-мат по-фински из говядины и сельди хранится 2 дня. Второй пользователь спрашивает о части рыбы в рецепте. Третий пользователь просит подсказку по жидкости для тонких блинов на сыворотке с содой.
